In [ ]:
import pandas as pd
data='/kaggle/input/hmisogyny/final_labels.csv'

In [ ]:
df = pd.read_csv(data)

In [ ]:
df = df.rename(columns={"level_3": "label"})

In [ ]:
df.head()

In [ ]:
df['label'] = df['label'].apply(lambda x: 1 if x == "Misogynistic" else 0)

In [ ]:
df_train = df[df['split']=='train']
df_test = df[df['split']=='test']

In [ ]:
df_train_itc=df_train[df_train['strength']=='Nature of the abuse is Implicit']
df_train_etc=df_train[df_train['strength']=='Nature of the abuse is Explicit']

In [ ]:
X=df_train['body']
y=df_train['label']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42, stratify=y)
df_train = pd.DataFrame({"body": X_train, "label": y_train})

In [ ]:
def get_qc_examples(df):
    """Creates examples for the training and dev sets."""
    text_and_labels = list(zip(df['body'], df['label']))
    return text_and_labels[1:]

In [ ]:
train_data = get_qc_examples(df_train)
test_data = get_qc_examples(df_test)

In [ ]:
unlabeled_examples=X_test
labelled_examples=train_data

In [ ]:
len(labelled_examples)

In [ ]:
len(unlabeled_examples)

In [ ]:
from collections import Counter
train_counts = Counter(label for _, label in labelled_examples)
print("Labeled Examples Count:", train_counts)
# Count labels in test examples
test_counts = Counter(label for _, label in test_data)
print("Test Examples Count:", test_counts)

In [ ]:
import torch
import io
import torch.nn.functional as F
import torch.nn as nn
import random
import numpy as np
import time
import math
import pandas as pd
import datetime
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [ ]:
import transformers
from transformers import *

In [ ]:
print(transformers.__version__)

In [ ]:
seed_val = 72
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seed_val)

In [ ]:
max_seq_length = 256
batch_size = 16
noise_size = 100
out_dropout_rate = 0.2
apply_balance = True
learning_rate_discriminator = 3e-7
learning_rate_generator = 6e-4
learning_rate_classifer = 5e-5
epsilon = 1e-8
num_train_epochs = 15
multi_gpu = True
apply_scheduler = False
warmup_proportion = 0.01
print_each_n_step = 100
num_labels=2
model_name="google-bert/bert-base-uncased"
label_list = [0,1]

In [ ]:
if torch.cuda.is_available():    
    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))
# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
transformer = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(transformer,tokenizer)

In [ ]:
def generate_data_loader(examples, do_shuffle = False, balance_label_examples = False):
  '''
  Generate a Dataloader given the input examples, eventually masked if they are 
  to be considered NOT labeled.
  '''

  input_ids = []
  input_mask_array = []
  label_id_array = []

  # Tokenization 
  for (text, label) in examples:
    encoded_sent = tokenizer.encode(str(text), add_special_tokens=True, max_length=max_seq_length, padding="max_length", truncation=True)
    input_ids.append(encoded_sent)
    label_id_array.append(int(label))
  
  # Attention to token (to ignore padded input wordpieces)
  for sent in input_ids:
    att_mask = [int(token_id > 0) for token_id in sent]                          
    input_mask_array.append(att_mask)
  # Convertion to Tensor
  input_ids = torch.tensor(input_ids) 
  input_mask_array = torch.tensor(input_mask_array)
  label_id_array = torch.tensor(label_id_array, dtype=torch.long)

  # Building the TensorDataset
  dataset = TensorDataset(input_ids, input_mask_array, label_id_array)

  if do_shuffle:
    sampler = RandomSampler
  else:
    sampler = SequentialSampler

  # Building the DataLoader
  return DataLoader(
              dataset,  # The training samples.
              sampler = sampler(dataset), 
              batch_size = batch_size) # Trains with this batch size.

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
def generate_unlabel_data_loader(input_examples, do_shuffle = False, balance_label_examples = False):
  '''
  Generate a Dataloader given the input examples, eventually masked if they are 
  to be considered NOT labeled.
  '''
  # examples = []

  # # if required it applies the balance
  # for index, ex in enumerate(input_examples): 
  #     examples.append(ex)
  #-----------------------------------------------
  # Generate input examples to the Transformer
  #-----------------------------------------------
  input_ids = []
  input_mask_array = []

  # Tokenization 
  for text in input_examples:
    encoded_sent = tokenizer.encode(str(text), add_special_tokens=True, max_length=max_seq_length, padding="max_length", truncation=True)
    input_ids.append(encoded_sent)
  
  # Attention to token (to ignore padded input wordpieces)
  for sent in input_ids:
    att_mask = [int(token_id > 0) for token_id in sent]                          
    input_mask_array.append(att_mask)
  # Convertion to Tensor
  input_ids = torch.tensor(input_ids) 
  input_mask_array = torch.tensor(input_mask_array)

  # Building the TensorDataset
  dataset = TensorDataset(input_ids, input_mask_array)

  if do_shuffle:
    sampler = RandomSampler
  else:
    sampler = SequentialSampler

  # Building the DataLoader
  return DataLoader(
              dataset,  # The training samples.
              sampler = sampler(dataset), 
              batch_size = batch_size) # Trains with this batch size.

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
unlabel_dataloader=generate_unlabel_data_loader(unlabeled_examples,do_shuffle = True,balance_label_examples = apply_balance)
label_dataloader=generate_data_loader(labelled_examples,do_shuffle = True,balance_label_examples = apply_balance)
test_dataloader=generate_data_loader(test_data,do_shuffle = True,balance_label_examples = apply_balance)

In [ ]:
arr=[]
count=0
for batch in unlabel_dataloader:
    input_ids, attention_mask = batch
    # print(input_ids.shape)
    count+=1
    if torch.isnan(input_ids).any():
        print("NaN values in input_ids:", torch.isnan(input_ids).any())
    arr.append(len(input_ids))
print(max(arr))
print(count)

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_size=100, output_size=512, hidden_dim=768, num_labels=3, dropout_rate=0.1):
        super().__init__()
        
        self.model = nn.Sequential(
            # Combine noise and label embedding
            nn.Linear(noise_size*2, hidden_dim ),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),

            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, output_size),
            nn.Tanh()
        )
        
        self.label_embed = nn.Embedding(num_labels, noise_size)
        
    def forward(self, z, labels):
        # z: (batch, noise_dim), labels: (batch,)
        label_emb = self.label_embed(labels)  # (batch, hidden_dim)
        combined = torch.cat([z, label_emb], dim=1)  # (batch, noise_dim + hidden_dim)
        output = self.model(combined)  # (batch, output_size)
        output =torch.nan_to_num(output)
        return output, labels



In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_size=512, hidden_dim=256, num_labels=3, dropout_rate=0.1):
        super().__init__()
        
        self.label_embed = nn.Embedding(num_labels, input_size)
        self.noise_proj = nn.Linear(input_size, hidden_dim)
        self.dropout=nn.Dropout(p=dropout_rate)
        self.model = nn.Sequential(
            # Combine input and label embedding
            nn.Linear( input_size*2, hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim , hidden_dim//2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout_rate),
        )
        self.fc= nn.Linear(hidden_dim//2, 1)
        self.activate=nn.Sigmoid()
        
    def forward(self, inputs, labels):
        # inputs: (batch, input_size), labels: (batch,)
        inputs = inputs.float()
        label_emb = self.label_embed(labels)  # (batch, hidden_dim)
        combined = torch.cat([inputs, label_emb], dim=1)  # (batch, input_size + hidden_dim)  # Store the features for potential use
        features = self.model(combined)  # (batch, 1)
        out=self.fc(features)
        output=torch.sigmoid(out)
        return output

In [ ]:
class Classifer(nn.Module):
    def __init__(self,transformer=transformer,dropout=0.1):
        super().__init__()
        self.bert = transformer
        self.dropout=nn.Dropout(p=dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids,attention_mask=None):
        
        cls_embed = self.bert(input_ids,attention_mask).last_hidden_state[:, 0, :]
        
        embed=self.fc(cls_embed)
        
        probs=torch.softmax(embed, dim=1)
        
        max_probs, max_indices = torch.max(probs, dim=1)
        return input_ids,probs,max_probs,max_indices

In [ ]:
config = AutoConfig.from_pretrained(model_name)
hidden_size = int(config.hidden_size)
#-------------------------------------------------
#   Instantiate the Generator and Discriminator ,Classifier
#-------------------------------------------------
generator = Generator(noise_size=noise_size, output_size=256, hidden_dim=hidden_size, num_labels=2, dropout_rate=out_dropout_rate)
discriminator = Discriminator(input_size=256, hidden_dim=256, num_labels=2, dropout_rate=out_dropout_rate)
classifer=Classifer(transformer,dropout=0.5)
# Put everything in the GPU if available
if torch.cuda.is_available():    
  generator.cuda()
  discriminator.cuda()
  classifer.cuda()
  if multi_gpu:
    transformer = torch.nn.DataParallel(transformer)

In [ ]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

In [ ]:
alpha=0.05

In [ ]:
import time
import torch
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
import numpy as np
from sklearn.metrics import f1_score

# ------------------------------
# Setup training statistics and timers
# ------------------------------
training_stats = []
total_t0 = time.time()

# ------------------------------
# Define model parameter lists (fixed typo in discriminator parameters list)
# ------------------------------
c_vars = [p for p in classifer.parameters()]
d_vars = [p for p in discriminator.parameters()]
g_vars = [p for p in generator.parameters()]

c_p = sum(p.numel() for p in classifer.parameters() if p.requires_grad)
d_p = sum(p.numel() for p in discriminator.parameters() if p.requires_grad)
g_p = sum(p.numel() for p in generator.parameters() if p.requires_grad)
total_params = c_p+d_p+g_p
print("total parameters:",total_params)
# ------------------------------
# Create Optimizers
# ------------------------------
dis_optimizer = torch.optim.AdamW(d_vars, lr=learning_rate_discriminator, weight_decay=1e-4)
gen_optimizer = torch.optim.AdamW(g_vars, lr=learning_rate_generator, weight_decay=1e-4) 
cls_optimizer = torch.optim.AdamW(c_vars, lr=learning_rate_classifer, weight_decay=1e-4) 

# ------------------------------
# Setup Learning Rate Schedulers (if applicable)
# ------------------------------
if apply_scheduler:
    num_train_examples = len(train_labeled_examples) + len(train_unlabeled_examples)
    num_train_steps = int(num_train_examples / batch_size * num_train_epochs)
    num_warmup_steps = int(num_train_steps * warmup_proportion)
    
    scheduler_d = get_constant_schedule_with_warmup(dis_optimizer, num_warmup_steps=num_warmup_steps)
    scheduler_g = get_constant_schedule_with_warmup(gen_optimizer, num_warmup_steps=num_warmup_steps)
    scheduler_c = get_constant_schedule_with_warmup(cls_optimizer, num_warmup_steps=num_warmup_steps)

# ------------------------------
# Training Loop
# ------------------------------
best_f1 = 0.0
best_model_path = "best_triple_gan_classifier.pt"
for epoch_i in range(num_train_epochs):
    print(f'\n======== Epoch {epoch_i + 1} / {num_train_epochs} ========')
    print('Training...')
    
    t0 = time.time()
    tr_g_loss = 0
    tr_d_loss = 0
    tr_c_loss = 0
    
    # Set models to training mode
    classifer.train()
    generator.train()
    discriminator.train()
    
    # Create iterator for unlabeled data
    unlabeled_iterator = iter(unlabel_dataloader)
    
    # Iterate over labeled data batches
    for step, labeled_batch in enumerate(label_dataloader):
        # Print progress updates every few steps
        if step % print_each_n_step == 0 and step != 0:
            elapsed = format_time(time.time() - t0)
            print(f'  Batch {step:>5,}  of  {len(label_dataloader):>5,}.    Elapsed: {elapsed}.')
        
        # Get unlabeled batch (reset iterator if needed)
        try:
            unlabeled_batch = next(unlabeled_iterator)
        except StopIteration:
            unlabeled_iterator = iter(unlabel_dataloader)
            unlabeled_batch = next(unlabeled_iterator)
            
        # ------------------------------
        # Process Labeled Batch
        # ------------------------------
        b_input_ids   = labeled_batch[0].to(device)
        b_input_mask  = labeled_batch[1].to(device)
        b_labels      = labeled_batch[2].to(device)
        
        # ------------------------------
        # Process Unlabeled Batch
        # ------------------------------
        u_input_ids   = unlabeled_batch[0].to(device)
        u_input_mask  = unlabeled_batch[1].to(device)
        real_batch_size = b_input_ids.size(0)
        unlabeled_batch_size = u_input_ids.size(0)
        
        # ------------------------------
        # Forward Pass for Labeled Data through the Classifier
        # ------------------------------
        input_id, probs, max_prob, predlabel = classifer(b_input_ids, b_input_mask)
        
        # ------------------------------
        # Forward Pass for Unlabeled Data through the Classifier
        # ------------------------------
        u_input_id, u_probs, u_max_prob, u_predlabel = classifer(u_input_ids, u_input_mask)
        
        # ------------------------------
        # Generate Fake Data using the Generator
        # ------------------------------
        z = torch.randn(real_batch_size, noise_size).to(device)
        fake_labels = torch.randint(0, num_labels, (real_batch_size,)).to(device)
        gen_rep, fakelabel = generator(z, fake_labels)
        
        # ------------------------------
        # Discriminator Forward Passes
        # ------------------------------
        # For unlabeled real data
        u_cls_out = discriminator(u_input_id.detach(), u_predlabel)
        # For generated fake data
        # print(u_cls_out)
        gen_out = discriminator(gen_rep.detach(), fakelabel)
        # For labeled real data
        # print(gen_out
        # print(torch.isnan(input_id).any(), torch.isinf(input_id).any())
        # print(torch.isnan(b_labels).any(), torch.isinf(b_labels).any())

        real_out = discriminator(input_id, b_labels)
        # print(real_out)
        
        # ------------------------------
        # Loss Calculations
        # ------------------------------
        # Generator Loss: adversarial loss from unlabeled data
        adv_loss = torch.mean(F.binary_cross_entropy(
            torch.sigmoid(gen_out),
            torch.ones_like(gen_out)
        ))
        g_loss = (1 - alpha) * adv_loss
        
        # Discriminator Loss
        d_loss_r = torch.mean(F.binary_cross_entropy(real_out, torch.ones_like(real_out)))
        dg_adv_loss = torch.mean(F.binary_cross_entropy(
            torch.sigmoid(gen_out),
            torch.ones_like(gen_out)
        ))
        d_loss_g = (1 - alpha) * dg_adv_loss 
        dc_adv_loss = torch.mean(F.binary_cross_entropy(
            torch.sigmoid(u_cls_out),
            torch.ones_like(u_cls_out)
        ))
        d_loss_c = (1 - alpha) * dc_adv_loss  # Additional loss for unlabeled data
        d_loss = d_loss_r + d_loss_g + d_loss_c
        
        # Classifier Loss (on labeled data)
        gen_rep_detached = gen_rep.detach()
        # (Assuming the classifier accepts integer tensors if needed; adjust as necessary)
        gen_input, gen_probs, _, genlabel = classifer(gen_rep_detached.long())
        target = F.one_hot(b_labels, num_classes=probs.size(1)).float()
        target_fake = F.one_hot(fakelabel, num_classes=num_labels).float()
        Rl = F.kl_div(torch.log(probs + 1e-8), target, reduction='batchmean')
        Rp = F.kl_div(torch.log(gen_probs + 1e-8), target_fake, reduction='batchmean')
        
        # Additional loss for unlabeled data (using probabilities of the true class)
        probs_of_true_class = u_probs[torch.arange(u_probs.size(0)), u_predlabel]
        cd_adv_loss = torch.mean(probs_of_true_class * F.binary_cross_entropy(
            torch.sigmoid(u_cls_out),
            torch.ones_like(u_cls_out)
        ))
        # supcon_loss = supervised_contrastive_loss(embeddings, b_labels, temperature=0.1)
        c_loss = (alpha * cd_adv_loss) + Rl + 0.1 * Rp 
        
        # ------------------------------
        # Optimization: Zero gradients and backpropagate
        # ------------------------------
        gen_optimizer.zero_grad()
        dis_optimizer.zero_grad()
        cls_optimizer.zero_grad()
        
        d_loss.backward(retain_graph=True)
        
        # Generator update: two steps
        for _ in range(2):
            gen_optimizer.zero_grad()
            g_loss.backward(retain_graph=True)
            gen_optimizer.step()
            
        c_loss.backward()
        
        dis_optimizer.step()
        cls_optimizer.step()
        
        # ------------------------------
        # Track Losses
        # ------------------------------
        tr_g_loss += g_loss.item()
        tr_d_loss += d_loss.item()
        tr_c_loss += c_loss.item()
        
        if apply_scheduler:
            scheduler_d.step()
            scheduler_g.step()
            scheduler_c.step()
    
    # ------------------------------
    # Average Loss Calculation for Epoch
    # ------------------------------
    avg_train_loss_g = tr_g_loss / len(label_dataloader)
    avg_train_loss_d = tr_d_loss / len(label_dataloader)             
    avg_train_loss_c = tr_c_loss / len(label_dataloader)
    training_time = format_time(time.time() - t0)
    
    print(f"\n  Average training loss generator: {avg_train_loss_g:.3f}")
    print(f"  Average training loss discriminator: {avg_train_loss_d:.3f}")
    print(f"  Average training loss classifier: {avg_train_loss_c:.3f}")
    print(f"  Training epoch took: {training_time}")
    
    # ------------------------------
    # Evaluation Phase
    # ------------------------------
    print("\nRunning Test...")
    t0 = time.time()

    # Set models to evaluation mode
    classifer.eval()
    discriminator.eval()
    generator.eval()

    total_test_loss = 0
    all_preds = []
    all_labels_ids = []

    nll_loss = CrossEntropyLoss(ignore_index=-1)

    for batch in test_dataloader:
        b_input_ids  = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels     = batch[2].to(device)
        
        with torch.no_grad():        
            # Ensure consistent naming for arguments (using keyword if needed)
            _, probs, _, _ = classifer(b_input_ids, b_input_mask)
            total_test_loss += nll_loss(probs, b_labels)
            
        _, preds = torch.max(probs, 1)
        all_preds.append(preds.detach().cpu())
        all_labels_ids.append(b_labels.detach().cpu())

   
    all_preds = torch.cat(all_preds).numpy()
    all_labels_ids = torch.cat(all_labels_ids).numpy()
    test_accuracy = np.sum(all_preds == all_labels_ids) / len(all_preds)
    print(f"  Accuracy: {test_accuracy:.3f}")

    macro_f1 = f1_score(all_labels_ids, all_preds, average='macro')
    print(f"F1 Score: {macro_f1:.3f}")

    # Save best classifier based on F1 score
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(classifer.state_dict(), best_model_path)
        print(f"\n✅ Saved new best model at Epoch {epoch_i + 1} with F1: {best_f1:.4f}")

    
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_loss = avg_test_loss.item()
    test_time = format_time(time.time() - t0)
    
    print(f"  val Loss: {avg_test_loss:.3f}")
    print(f"  val took: {test_time}")

    # ------------------------------
    # Record Statistics for the Epoch
    # ------------------------------
    training_stats.append({
        'epoch': epoch_i + 1,
        'Training Loss generator': avg_train_loss_g,
        'Training Loss discriminator': avg_train_loss_d,
        'Training Loss Classifier': avg_train_loss_c,
        'Valid. Loss': avg_test_loss,
        'Valid. Accur.': test_accuracy,
        'F1 Score': macro_f1,
        'Training Time': training_time,
        'Test Time': test_time
    })


In [ ]:
import matplotlib.pyplot as plt
epochs = [entry['epoch'] for entry in training_stats]  
accuracy = [entry['Valid. Accur.'] for entry in training_stats]  
f1_scores = [entry['F1 Score'] for entry in training_stats]  
test_loss=[entry['Valid. Loss'] for entry in training_stats]
d_loss=[entry['Training Loss discriminator'] for entry in training_stats]
g_loss=[entry['Training Loss generator'] for entry in training_stats]
c_loss=[entry['Training Loss Classifier'] for entry in training_stats]
# Plot accuracy and F1 score
plt.figure(figsize=(10, 6))
plt.plot(epochs, accuracy, label="Accuracy", marker="o")
plt.plot(epochs, f1_scores, label="F1 Score", marker="o")

plt.title("Accuracy and F1 Score over Epochs", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Metrics", fontsize=12)
plt.xticks(epochs)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
plt.plot(epochs, test_loss, label="test_loss", marker="o")
# plt.plot(epochs,d_loss , label="d_loss", marker="x")
plt.plot(epochs,g_loss , label="g_loss", marker="x")
plt.plot(epochs,c_loss , label="c_loss", marker="x")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Metrics", fontsize=12)
plt.xticks(epochs)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
classifer.load_state_dict(torch.load("best_triple_gan_classifier.pt", weights_only=True))
classifer.eval()

In [ ]:
from lime.lime_text import LimeTextExplainer
import torch.nn.functional as F

def preprocess_texts(texts):
    
    input_ids = []
    attention_masks = []
    for text in texts:
        encoding = tokenizer.encode_plus(text,
                                         max_length=max_seq_length,
                                         truncation=True,
                                         padding='max_length',
                                         return_tensors='pt')
        input_ids.append(encoding['input_ids'])
        attention_masks.append(encoding['attention_mask'])
    
    input_ids = torch.cat(input_ids).to(device)
    attention_masks = torch.cat(attention_masks).to(device)
    return input_ids, attention_masks


def predict_fn(texts, batch_size=16):
    all_probs = []
    classifer.eval()
    print(len(texts))
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        input_ids, attention_mask = preprocess_texts(batch)
        with torch.no_grad():
            out, probs, max_probs, _ = classifer(input_ids, attention_mask)
            # print(max_probs)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)


In [ ]:
# Replace with your class names
class_names = ['non-misogyny', 'misogyny']
explainer = LimeTextExplainer(class_names=class_names)

# Pick a test example
sample_text = "Women are too emotional to be good leaders."

# Explain the prediction
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10,num_samples=100)
explanation.show_in_notebook(text=True)


In [ ]:
sample_text = "Women just aren’t as interested in tech careers"
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10,num_samples=100)
explanation.show_in_notebook(text=True)

In [ ]:
sample_text = "Women are already equal now — there's no need for feminism anymore"
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10,num_samples=100)
explanation.show_in_notebook(text=True)

In [ ]:
# Men are the real victims these days — everything is about empowering women now.
sample_text = "Men are the real victims these days because everything is about empowering women now."
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10,num_samples=100)
explanation.show_in_notebook(text=True)

In [ ]:

sample_text = "There aren’t many things that are more satisfying than telling a girl, 'No' "
explanation = explainer.explain_instance(sample_text, predict_fn, num_features=10,num_samples=100)
explanation.show_in_notebook(text=True)